In [ ]:
!git clone https://github.com/pratikkayal/PlantDoc-Dataset.git

Cloning into 'PlantDoc-Dataset'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 28.81 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.


In [ ]:
from pathlib import Path

train_path = Path("PlantDoc-Dataset/train")
test_path  = Path("PlantDoc-Dataset/test")

train_folders = sorted([f.name for f in train_path.iterdir() if f.is_dir()])
print(f"Train folders ({len(train_folders)}):")
for f in train_folders:
    count = len(list((train_path / f).glob("*.*")))
    print(f"  {f:<45} → {count} images")

Train folders (28):
  Apple Scab Leaf                               → 83 images
  Apple leaf                                    → 82 images
  Apple rust leaf                               → 79 images
  Bell_pepper leaf                              → 53 images
  Bell_pepper leaf spot                         → 62 images
  Blueberry leaf                                → 106 images
  Cherry leaf                                   → 47 images
  Corn Gray leaf spot                           → 64 images
  Corn leaf blight                              → 180 images
  Corn rust leaf                                → 106 images
  Peach leaf                                    → 103 images
  Potato leaf early blight                      → 109 images
  Potato leaf late blight                       → 97 images
  Raspberry leaf                                → 112 images
  Soyabean leaf                                 → 57 images
  Squash Powdery mildew leaf                    → 124 images
  Strawberry 

In [ ]:
train_imgs = list(train_path.rglob("*.jpg")) + list(train_path.rglob("*.png"))
test_imgs  = list(test_path.rglob("*.jpg"))  + list(test_path.rglob("*.png"))

print(f"Train images : {len(train_imgs)}")
print(f"Test images  : {len(test_imgs)}")
print(f"Total        : {len(train_imgs) + len(test_imgs)}")

Train images : 2341
Test images  : 236
Total        : 2577


In [ ]:
# Authenticate first, then mount
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install torch torchvision transformers -q

In [ ]:
train_folders = sorted([f.name for f in Path("PlantDoc-Dataset/train").iterdir() if f.is_dir()])
for f in train_folders:
    print(f'    "{f}": "",')

    "Apple Scab Leaf": "",
    "Apple leaf": "",
    "Apple rust leaf": "",
    "Bell_pepper leaf": "",
    "Bell_pepper leaf spot": "",
    "Blueberry leaf": "",
    "Cherry leaf": "",
    "Corn Gray leaf spot": "",
    "Corn leaf blight": "",
    "Corn rust leaf": "",
    "Peach leaf": "",
    "Potato leaf early blight": "",
    "Potato leaf late blight": "",
    "Raspberry leaf": "",
    "Soyabean leaf": "",
    "Squash Powdery mildew leaf": "",
    "Strawberry leaf": "",
    "Tomato Early blight leaf": "",
    "Tomato Septoria leaf spot": "",
    "Tomato leaf": "",
    "Tomato leaf bacterial spot": "",
    "Tomato leaf late blight": "",
    "Tomato leaf mosaic virus": "",
    "Tomato leaf yellow virus": "",
    "Tomato mold leaf": "",
    "Tomato two spotted spider mites leaf": "",
    "grape leaf": "",
    "grape leaf black rot": "",


In [ ]:
from pathlib import Path

train_path = Path("PlantDoc-Dataset/train")
test_path  = Path("PlantDoc-Dataset/test")

train_folders = sorted([f.name for f in train_path.iterdir() if f.is_dir()])
print(f"Train folders ({len(train_folders)}):")
for f in train_folders:
    count = len(list((train_path / f).glob("*.*")))
    print(f"  {f:<45} → {count} images")

Train folders (28):
  Apple Scab Leaf                               → 83 images
  Apple leaf                                    → 82 images
  Apple rust leaf                               → 79 images
  Bell_pepper leaf                              → 53 images
  Bell_pepper leaf spot                         → 62 images
  Blueberry leaf                                → 106 images
  Cherry leaf                                   → 47 images
  Corn Gray leaf spot                           → 64 images
  Corn leaf blight                              → 180 images
  Corn rust leaf                                → 106 images
  Peach leaf                                    → 103 images
  Potato leaf early blight                      → 109 images
  Potato leaf late blight                       → 97 images
  Raspberry leaf                                → 112 images
  Soyabean leaf                                 → 57 images
  Squash Powdery mildew leaf                    → 124 images
  Strawberry 

In [ ]:
PLANTDOC_SPECIES_MAP = {
    "Apple Scab Leaf":                      "Apple",
    "Apple leaf":                           "Apple",
    "Apple rust leaf":                      "Apple",
    "Bell_pepper leaf":                     "Bell Pepper",
    "Bell_pepper leaf spot":                "Bell Pepper",
    "Blueberry leaf":                       "Blueberry",
    "Cherry leaf":                          "Cherry",
    "Corn Gray leaf spot":                  "Corn",
    "Corn leaf blight":                     "Corn",
    "Corn rust leaf":                       "Corn",
    "Peach leaf":                           "Peach",
    "Potato leaf early blight":             "Potato",
    "Potato leaf late blight":              "Potato",
    "Raspberry leaf":                       "Raspberry",
    "Soyabean leaf":                        "Soybean",
    "Squash Powdery mildew leaf":           "Squash",
    "Strawberry leaf":                      "Strawberry",
    "Tomato Early blight leaf":             "Tomato",
    "Tomato Septoria leaf spot":            "Tomato",
    "Tomato leaf":                          "Tomato",
    "Tomato leaf bacterial spot":           "Tomato",
    "Tomato leaf late blight":              "Tomato",
    "Tomato leaf mosaic virus":             "Tomato",
    "Tomato leaf yellow virus":             "Tomato",
    "Tomato mold leaf":                     "Tomato",
    "Tomato two spotted spider mites leaf": "Tomato",
    "grape leaf":                           "Grape",
    "grape leaf black rot":                 "Grape",
}

# Verify
all_folders = [f.name for f in train_path.iterdir() if f.is_dir()]
missing = [f for f in all_folders if f not in PLANTDOC_SPECIES_MAP]
print(f"Total mapped  : {len(PLANTDOC_SPECIES_MAP)}/28")
print(f"Unique species: {sorted(set(PLANTDOC_SPECIES_MAP.values()))}")
print(f"Missing       : {missing if missing else '✅ None'}")

Total mapped  : 28/28
Unique species: ['Apple', 'Bell Pepper', 'Blueberry', 'Cherry', 'Corn', 'Grape', 'Peach', 'Potato', 'Raspberry', 'Soybean', 'Squash', 'Strawberry', 'Tomato']
Missing       : ✅ None


In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class PlantDocSpeciesDataset(Dataset):
    def __init__(self, root_dir, species_map, transform=None):
        self.transform = transform
        self.samples   = []

        all_species          = sorted(set(species_map.values()))
        self.species_to_idx  = {s: i for i, s in enumerate(all_species)}
        self.idx_to_species  = {i: s for s, i in self.species_to_idx.items()}
        self.num_classes     = len(all_species)

        skipped = []
        for folder in Path(root_dir).iterdir():
            if not folder.is_dir():
                continue
            species = species_map.get(folder.name)
            if species is None:
                skipped.append(folder.name)
                continue
            label = self.species_to_idx[species]
            for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG"]:
                for img_path in folder.glob(ext):
                    self.samples.append((img_path, label))

        print(f"✅ Loaded {len(self.samples)} images | {self.num_classes} species")
        if skipped:
            print(f"⚠️  Skipped: {skipped}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

    def class_distribution(self):
        from collections import Counter
        counts = Counter(label for _, label in self.samples)
        return {self.idx_to_species[k]: v for k, v in sorted(counts.items())}

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("✅ Transforms ready")


✅ Transforms ready


In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter

train_ds = PlantDocSpeciesDataset(
    "PlantDoc-Dataset/train", PLANTDOC_SPECIES_MAP, train_transform
)
val_ds = PlantDocSpeciesDataset(
    "PlantDoc-Dataset/test", PLANTDOC_SPECIES_MAP, val_transform
)

print("\nClass distribution (train):")
for species, count in train_ds.class_distribution().items():
    print(f"  {species:<15} → {count} images")

# Weighted sampler for class imbalance
label_counts = Counter(label for _, label in train_ds.samples)
weights      = [1.0 / label_counts[label] for _, label in train_ds.samples]
sampler      = WeightedRandomSampler(weights,
                                     num_samples=len(weights),
                                     replacement=True)

train_loader = DataLoader(train_ds, batch_size=32,
                          sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32,
                          shuffle=False, num_workers=2)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

✅ Loaded 2342 images | 13 species
✅ Loaded 236 images | 13 species

Class distribution (train):
  Apple           → 244 images
  Bell Pepper     → 115 images
  Blueberry       → 106 images
  Cherry          → 47 images
  Corn            → 350 images
  Grape           → 113 images
  Peach           → 103 images
  Potato          → 206 images
  Raspberry       → 112 images
  Soybean         → 57 images
  Squash          → 124 images
  Strawberry      → 88 images
  Tomato          → 677 images

Train batches : 74
Val batches   : 8


In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter

train_ds = PlantDocSpeciesDataset(
    "PlantDoc-Dataset/train", PLANTDOC_SPECIES_MAP, train_transform
)
val_ds = PlantDocSpeciesDataset(
    "PlantDoc-Dataset/test", PLANTDOC_SPECIES_MAP, val_transform
)

print("\nClass distribution (train):")
for species, count in train_ds.class_distribution().items():
    print(f"  {species:<15} → {count} images")

# Weighted sampler for class imbalance
label_counts = Counter(label for _, label in train_ds.samples)
weights      = [1.0 / label_counts[label] for _, label in train_ds.samples]
sampler      = WeightedRandomSampler(weights,
                                     num_samples=len(weights),
                                     replacement=True)

train_loader = DataLoader(train_ds, batch_size=32,
                          sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32,
                          shuffle=False, num_workers=2)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

✅ Loaded 2342 images | 13 species
✅ Loaded 236 images | 13 species

Class distribution (train):
  Apple           → 244 images
  Bell Pepper     → 115 images
  Blueberry       → 106 images
  Cherry          → 47 images
  Corn            → 350 images
  Grape           → 113 images
  Peach           → 103 images
  Potato          → 206 images
  Raspberry       → 112 images
  Soybean         → 57 images
  Squash          → 124 images
  Strawberry      → 88 images
  Tomato          → 677 images

Train batches : 74
Val batches   : 8


In [ ]:
import torch.nn as nn
from transformers import AutoModel

class DINOv2SpeciesClassifier(nn.Module):
    def __init__(self, num_species, freeze_backbone=True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("facebook/dinov2-base")

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        hidden_dim = self.backbone.config.hidden_size  # 768

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_species)
        )

    def forward(self, pixel_values):
        outputs  = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_token)

    def extract_features(self, pixel_values):
        """Returns raw 768-d embedding — used later by cache module"""
        with torch.no_grad():
            outputs = self.backbone(pixel_values=pixel_values)
        return outputs.last_hidden_state[:, 0, :]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = DINOv2SpeciesClassifier(
    num_species=train_ds.num_classes,
    freeze_backbone=True
).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}  (classifier head only)")
print(f"Species classes : {train_ds.num_classes}")
print(f"Species list    : {list(train_ds.species_to_idx.keys())}")

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Total params    : 86,782,221
Trainable params: 201,741  (classifier head only)
Species classes : 13
Species list    : ['Apple', 'Bell Pepper', 'Blueberry', 'Cherry', 'Corn', 'Grape', 'Peach', 'Potato', 'Raspberry', 'Soybean', 'Squash', 'Strawberry', 'Tomato']


In [ ]:
optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=1e-3,
    weight_decay=0.01
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=30
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print("✅ Optimizer, scheduler, loss ready")

✅ Optimizer, scheduler, loss ready


In [ ]:
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits     = model(images)
            total_loss += criterion(logits, labels).item()
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total

EPOCHS       = 30
best_val_acc = 0.0

print(f"Starting training for {EPOCHS} epochs...\n")

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader,
                                        optimizer, criterion, device)
    val_loss,   val_acc   = validate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch":            epoch + 1,
            "model_state":      model.state_dict(),
            "optimizer_state":  optimizer.state_dict(),
            "val_acc":          val_acc,
            "species_to_idx":   train_ds.species_to_idx,
            "idx_to_species":   train_ds.idx_to_species,
        }, f"{CHECKPOINT_DIR}/dinov2_species_best.pt")
        print(f"  ✅ Best model saved (val_acc={val_acc:.3f})")

print(f"\n🎉 Training complete | Best val acc: {best_val_acc:.3f}")

Starting training for 30 epochs...

Epoch 01/30 | Train Loss: 0.9522  Acc: 0.876 | Val Loss: 0.8016  Acc: 0.924
  ✅ Best model saved (val_acc=0.924)
Epoch 02/30 | Train Loss: 0.7153  Acc: 0.953 | Val Loss: 0.8312  Acc: 0.924
Epoch 03/30 | Train Loss: 0.7026  Acc: 0.953 | Val Loss: 0.7423  Acc: 0.941
  ✅ Best model saved (val_acc=0.941)
Epoch 04/30 | Train Loss: 0.6718  Acc: 0.968 | Val Loss: 0.7500  Acc: 0.941
Epoch 05/30 | Train Loss: 0.6781  Acc: 0.968 | Val Loss: 0.8094  Acc: 0.911
Epoch 06/30 | Train Loss: 0.6529  Acc: 0.973 | Val Loss: 0.7454  Acc: 0.941
Epoch 07/30 | Train Loss: 0.6502  Acc: 0.973 | Val Loss: 0.7179  Acc: 0.953
  ✅ Best model saved (val_acc=0.953)
Epoch 08/30 | Train Loss: 0.6486  Acc: 0.971 | Val Loss: 0.7913  Acc: 0.911
Epoch 09/30 | Train Loss: 0.6410  Acc: 0.977 | Val Loss: 0.7304  Acc: 0.945
Epoch 10/30 | Train Loss: 0.6290  Acc: 0.984 | Val Loss: 0.6975  Acc: 0.945
Epoch 11/30 | Train Loss: 0.6253  Acc: 0.985 | Val Loss: 0.7602  Acc: 0.941
Epoch 12/30 | Tra

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Load best model
checkpoint = torch.load(f"{CHECKPOINT_DIR}/dinov2_species_best.pt")
model.load_state_dict(checkpoint["model_state"])
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        logits = model(images)
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

# Per species report
species_names = [train_ds.idx_to_species[i] for i in range(train_ds.num_classes)]
print(classification_report(all_labels, all_preds,
                             target_names=species_names, digits=3))

              precision    recall  f1-score   support

       Apple      0.966     0.966     0.966        29
 Bell Pepper      0.773     1.000     0.872        17
   Blueberry      1.000     0.909     0.952        11
      Cherry      1.000     1.000     1.000        10
        Corn      1.000     1.000     1.000        26
       Grape      0.952     1.000     0.976        20
       Peach      1.000     1.000     1.000         9
      Potato      1.000     0.875     0.933        16
   Raspberry      0.875     1.000     0.933         7
     Soybean      1.000     0.625     0.769         8
      Squash      1.000     0.833     0.909         6
  Strawberry      1.000     1.000     1.000         8
      Tomato      0.971     0.971     0.971        69

    accuracy                          0.958       236
   macro avg      0.964     0.937     0.945       236
weighted avg      0.963     0.958     0.957       236



In [ ]:
from google.colab import files
files.download(f"{CHECKPOINT_DIR}/dinov2_species_best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from collections import defaultdict

# Maps each species to its disease folder names
SPECIES_TO_DISEASES = defaultdict(list)
for folder_name, species in PLANTDOC_SPECIES_MAP.items():
    SPECIES_TO_DISEASES[species].append(folder_name)

print("Species → Disease mapping:")
for species, diseases in sorted(SPECIES_TO_DISEASES.items()):
    print(f"\n  {species}:")
    for d in diseases:
        print(f"    - {d}")

Species → Disease mapping:

  Apple:
    - Apple Scab Leaf
    - Apple leaf
    - Apple rust leaf

  Bell Pepper:
    - Bell_pepper leaf
    - Bell_pepper leaf spot

  Blueberry:
    - Blueberry leaf

  Cherry:
    - Cherry leaf

  Corn:
    - Corn Gray leaf spot
    - Corn leaf blight
    - Corn rust leaf

  Grape:
    - grape leaf
    - grape leaf black rot

  Peach:
    - Peach leaf

  Potato:
    - Potato leaf early blight
    - Potato leaf late blight

  Raspberry:
    - Raspberry leaf

  Soybean:
    - Soyabean leaf

  Squash:
    - Squash Powdery mildew leaf

  Strawberry:
    - Strawberry leaf

  Tomato:
    - Tomato Early blight leaf
    - Tomato Septoria leaf spot
    - Tomato leaf
    - Tomato leaf bacterial spot
    - Tomato leaf late blight
    - Tomato leaf mosaic virus
    - Tomato leaf yellow virus
    - Tomato mold leaf
    - Tomato two spotted spider mites leaf
